In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import tensorflow as tf
from util import yolo_filter_boxes, yolo_non_max_suppression, yolo_eval, yolo_boxes_to_corners, scale_boxes
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
import matplotlib.pyplot as plt

### Find highest classes probability above threshold

In [ ]:
tf.random.set_seed(10)
box_confidence = tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([19, 19, 5, 4], mean=1, stddev=4, seed = 1)
box_class_probs = tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = 0.5)
print("\n")
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.shape))
print("boxes.shape = " + str(boxes.shape))
print("classes.shape = " + str(classes.shape))

### Apply Non Max Suppression to remove overlap anchorbox

In [ ]:
tf.random.set_seed(10)
scores = tf.random.normal([54,], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([54, 4], mean=1, stddev=4, seed = 1)
classes = tf.random.normal([54,], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_non_max_suppression(scores, boxes, classes)

print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

### Calculate box [ymin,xmin,ymax,xmax] by box center point x_y and box w_h

In [36]:
# W, H, num_anchors, x_y
# W, H, num_anchors, w_h
# W, H, num_anchors, confidence
# W, H, num_anchors, classes_probs
yolo_outputs = (tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1))

box_xy, box_wh, box_confidence, box_class_probs = yolo_outputs
# box_xy[:, 0]
boxes = yolo_boxes_to_corners(box_xy, box_wh)
# W, H, num_anchors, [ymin,xmin,ymax,xmax]
print(boxes.shape)
# print(boxes[:, 0])

(19, 19, 5, 4)


### Convert box coodinates into image shape

In [ ]:
scores, boxes, classes = yolo_filter_boxes(boxes, box_confidence, box_class_probs, 0.6)
print("boxes",boxes.shape) # hight class prob = [W,H,num_anchors, 1] = [W,H,num_anchors]
# print(boxes[:, 0]) 

# Scale boxes back to original image shape (720, 1280 or whatever)
boxes = scale_boxes(boxes, image_shape = (720, 1280))
print(boxes.shape)
# print(boxes[:, 0]) 

(19, 19, 5, 80)
box_classes (19, 19, 5)
box_classes_score (19, 19, 5)
filter (19, 19, 5) (19, 19, 5, 4)
boxes (1784, 4)
boxes.shape (1784, 4) image_dims (1, 4)
(1784, 4)


In [31]:
# 1. Create a dummy tensor of shape (19, 19, 5)
matrix = tf.random.uniform(shape=(19, 19, 5))

# 2. Create a boolean mask of the exact same shape (e.g., finding values > 0.5)
mask = matrix > 0.5

# 3. Apply the boolean mask
result = tf.boolean_mask(matrix, mask)

# 4. Check the resulting shape
print("Output Shape:", result.shape)

Output Shape: (885,)


### Convert yolo boxes into bounding boxes and apply filtering

In [18]:
tf.random.set_seed(10)
yolo_outputs = (tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1))
scores, boxes, classes = yolo_eval(yolo_outputs)
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

(19, 19, 5, 80)
box_classes (19, 19, 5)
box_classes_score (19, 19, 5)
filter (19, 19, 5)
scores[2] = 171.60194
boxes[2] = [-1240.3483 -3212.5881  -645.78    2024.3052]
classes[2] = 16
scores.shape = (10,)
boxes.shape = (10, 4)
classes.shape = (10,)
